# Notebook 1 — Database and SQL Basics
## Learn tables, keys, relationships, and SQL from zero

**Who is this for?** A learner who has never used a database.

**What you will learn**
1. What a database is and why applications need one.
2. Tables, rows, columns and data types.
3. Primary keys and foreign keys.
4. One-to-one, one-to-many and many-to-many relationships.
5. Basic SQL: `SELECT`, `WHERE`, `ORDER BY`, `INSERT`, `UPDATE`, `DELETE`, and `JOIN`.
6. How Django models relate to database tables.

**How to use this notebook**
- Read each explanation before running its code.
- Run cells from top to bottom.
- Read the output, not just the code.
- Try each exercise before opening its answer.
- This notebook uses Python's built-in `sqlite3` only for safe, local SQL practice. The main Django course uses PostgreSQL. SQL concepts transfer, but a few SQL details differ between database systems.

**Important:** This notebook is a learning sandbox. It does not connect to or change your Neon database.

## 1. What is a database?

A database is an organised place where an application stores information so it can find and use it later.

Imagine a school:
- Students have names and student numbers.
- Teachers have names and staff numbers.
- Classes have subjects and teachers.
- Students join classes.

A database stores these facts in a structured way.

### What is a table?
A table stores one kind of thing. A `students` table stores students.

### What is a row?
One row is one student record.

### What is a column?
A column is one type of information, such as `student_name`.

A spreadsheet is a useful first picture, but a real database also has rules, relationships, query tools and safe ways for many users to work at once.

In [ ]:
# A simple Python list is NOT a database, but it helps us picture records.
students = [
    {"id": 1, "name": "Asha", "age": 16},
    {"id": 2, "name": "Rahul", "age": 17},
]
students

The output is a list of two student records. If the Python program ends, this list is not automatically saved permanently. A database gives us persistent storage.

## 2. Create a tiny practice database

Python includes `sqlite3`, so we can practise SQL without creating an account or installing a database server. SQLite creates a local file. Our Django project uses PostgreSQL instead.

In [ ]:
import sqlite3

# Create a database file named day2_sql_practice.db in the notebook folder.
connection = sqlite3.connect("day2_sql_practice.db")
cursor = connection.cursor()

# Remove these practice tables if this notebook is run again.
cursor.execute("DROP TABLE IF EXISTS students")
cursor.execute("DROP TABLE IF EXISTS courses")

cursor.execute("""
CREATE TABLE students (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    age INTEGER NOT NULL
)
""")

cursor.execute("""
CREATE TABLE courses (
    id INTEGER PRIMARY KEY,
    course_name TEXT NOT NULL
)
""")

connection.commit()
print("Practice tables created.")

### Understand the SQL

- `CREATE TABLE` creates a table.
- `id` is an integer column.
- `PRIMARY KEY` makes `id` the unique identifier for each row.
- `TEXT` stores text.
- `NOT NULL` means a value is required.
- `INTEGER` stores whole numbers.
- `commit()` saves the database changes.

A primary key must identify a row uniquely. Two students can have the same name, but they must not have the same primary key.

## 3. Insert rows with INSERT

`INSERT` adds a new row to a table. We name the columns and provide their values.

Never build SQL by joining untrusted user text into the query. Use placeholders, as shown below.

In [ ]:
cursor.execute(
    "INSERT INTO students (id, name, age) VALUES (?, ?, ?)",
    (1, "Asha", 16)
)
cursor.execute(
    "INSERT INTO students (id, name, age) VALUES (?, ?, ?)",
    (2, "Rahul", 17)
)
cursor.execute(
    "INSERT INTO students (id, name, age) VALUES (?, ?, ?)",
    (3, "Meera", 16)
)
connection.commit()
print("Three students inserted.")

## 4. Read rows with SELECT

`SELECT` asks the database to return information.

- `SELECT *` means return all columns.
- `FROM students` says which table to read.

In [ ]:
cursor.execute("SELECT * FROM students")
rows = cursor.fetchall()
rows

Each inner tuple is one row. The values appear in the order of the columns.

### Select only the columns you need

In [ ]:
cursor.execute("SELECT name, age FROM students")
cursor.fetchall()

## 5. Filter rows with WHERE

`WHERE` keeps only rows that match a condition.

The following query finds students who are 16.

In [ ]:
cursor.execute(
    "SELECT id, name, age FROM students WHERE age = ?",
    (16,)
)
cursor.fetchall()

The `?` is a placeholder used by SQLite. The value `(16,)` is supplied separately. This is called a parameterised query.

### Useful comparison operators
- `=` equals
- `!=` or `<>` not equal
- `>` greater than
- `<` less than
- `>=` greater than or equal
- `<=` less than or equal

In [ ]:
cursor.execute(
    "SELECT name, age FROM students WHERE age >= ?",
    (17,)
)
cursor.fetchall()

## 6. Sort rows with ORDER BY

`ORDER BY` sorts the result. `ASC` means ascending. `DESC` means descending.

In [ ]:
cursor.execute("SELECT name, age FROM students ORDER BY age DESC, name ASC")
cursor.fetchall()

## 7. Change rows with UPDATE

`UPDATE` changes existing rows. Always use a careful `WHERE` condition. Without `WHERE`, every row may be changed.

In [ ]:
cursor.execute(
    "UPDATE students SET age = ? WHERE id = ?",
    (18, 2)
)
connection.commit()

cursor.execute("SELECT * FROM students ORDER BY id")
cursor.fetchall()

Rahul's age changed because the row with `id = 2` was selected.

## 8. Remove rows with DELETE

`DELETE` removes rows. Again, always check the `WHERE` condition first.

In [ ]:
# Delete only the practice record with id 3.
cursor.execute("DELETE FROM students WHERE id = ?", (3,))
connection.commit()

cursor.execute("SELECT * FROM students ORDER BY id")
cursor.fetchall()

## 9. What is a relationship?

A relationship connects records in different tables.

### Primary key
A primary key identifies one row in its own table.

### Foreign key
A foreign key stores a reference to a row in another table.

For example, a student may have a `course_id` that points to a course. This is a one-to-many example if each course can have many students and each student belongs to one course.

In [ ]:
# Create a new table that demonstrates a foreign key.
cursor.execute("DROP TABLE IF EXISTS enrolments")
cursor.execute("DROP TABLE IF EXISTS course_catalog")

cursor.execute("""
CREATE TABLE course_catalog (
    id INTEGER PRIMARY KEY,
    course_name TEXT NOT NULL
)
""")

cursor.execute("""
CREATE TABLE enrolments (
    id INTEGER PRIMARY KEY,
    student_id INTEGER NOT NULL,
    course_id INTEGER NOT NULL,
    FOREIGN KEY (student_id) REFERENCES students(id),
    FOREIGN KEY (course_id) REFERENCES course_catalog(id)
)
""")

cursor.execute("INSERT INTO course_catalog (id, course_name) VALUES (?, ?)", (1, "Python"))
cursor.execute("INSERT INTO course_catalog (id, course_name) VALUES (?, ?)", (2, "Django"))
connection.commit()
print("Related tables created.")

### One-to-One
One row on the first side connects to one row on the second side.

Example: one user has one profile. Django uses `OneToOneField`.

### One-to-Many
One parent row can connect to many child rows. Each child points to one parent.

Example: one author writes many articles. Django uses `ForeignKey` on the Article model.

### Many-to-Many
Many rows on both sides can connect to many rows on the other side.

Example: an article can have many categories, and a category can be used by many articles. Django uses `ManyToManyField` and creates a linking table.

Try drawing all three relationships in your notebook before continuing.

## 10. JOIN: read related information

A `JOIN` combines rows from connected tables. The join condition tells the database which key values match.

This query joins enrolments to course names.

In [ ]:
cursor.execute("""
SELECT enrolments.id, enrolments.student_id, course_catalog.course_name
FROM enrolments
JOIN course_catalog
    ON enrolments.course_id = course_catalog.id
""")
cursor.fetchall()

The result is empty because we have not inserted enrolment rows yet. Add some now.

In [ ]:
cursor.execute("INSERT INTO enrolments (id, student_id, course_id) VALUES (1, 1, 1)")
cursor.execute("INSERT INTO enrolments (id, student_id, course_id) VALUES (2, 2, 2)")
connection.commit()

cursor.execute("""
SELECT students.name, course_catalog.course_name
FROM enrolments
JOIN students ON enrolments.student_id = students.id
JOIN course_catalog ON enrolments.course_id = course_catalog.id
ORDER BY students.name
""")
cursor.fetchall()

## 11. SQL and Django ORM

Django ORM lets Python code work with database records using model classes.

| SQL idea | Django ORM example |
|---|---|
| Read all rows | `Article.objects.all()` |
| Filter rows | `Article.objects.filter(is_published=True)` |
| Find one row | `Article.objects.get(pk=1)` |
| Insert row | `Article.objects.create(title="Hello", ...)` |
| Change row | `article.title = "New"; article.save()` |
| Delete row | `article.delete()` |

The ORM does not remove the need to understand SQL. SQL helps you understand what the database is doing and diagnose performance problems.

## 12. Exercises — try before viewing answers

### Exercise 1
Write a query to show all students ordered by name.

### Exercise 2
Write a query to show students whose age is less than 18.

### Exercise 3
Change Asha's age to 17. Use her ID, not her name.

### Exercise 4
Explain the difference between a primary key and a foreign key.

### Exercise 5
Explain why an article-to-category relationship is many-to-many.

### Exercise 6
What could happen if you run `DELETE FROM students;` without a `WHERE` condition?

In [ ]:
# Exercise answers — run only after attempting the questions.

# 1
cursor.execute("SELECT * FROM students ORDER BY name ASC")
answer_1 = cursor.fetchall()

# 2
cursor.execute("SELECT * FROM students WHERE age < ?", (18,))
answer_2 = cursor.fetchall()

# 3
cursor.execute("UPDATE students SET age = ? WHERE id = ?", (17, 1))
connection.commit()

answer_1, answer_2

## 13. Clean up

Close the connection when you have finished. The database file remains in the notebook folder so you can practise again.

In [ ]:
connection.close()
print("Database connection closed.")

## 14. What you should now understand

You should be able to explain:
- Why applications need databases.
- Tables, rows and columns.
- Primary keys and foreign keys.
- One-to-one, one-to-many and many-to-many.
- How to read, insert, update and delete rows with SQL.
- How JOIN connects related tables.
- How Django ORM expresses common database operations.

**Next:** Open `02_Django_ORM_Practice.ipynb`, then follow the Django practical in the `django_lab/` folder.